# Introduction to PyTorch Lightning

[Link](https://lightning.ai/docs/pytorch/stable/notebooks/lightning_examples/mnist-hello-world.html#Introduction-to-PyTorch-Lightning)

Import modules.

In [1]:
import os
from dataclasses import dataclass
from typing import tuple

import pandas as pd
import pytorch_lightning as pl
import seaborn as sn
import torch

from IPython.display import display
from pytorch_lightning.loggers import CSVLogger
from pathlib import Path

from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torchmetrics import Accuracy
from torchvision import transforms
from torchvision.datasets import MNIST

ImportError: cannot import name 'tuple' from 'typing' (c:\Users\Mike\anaconda3\envs\gifteval\Lib\typing.py)

Create a config for our model.

In [ ]:
@dataclass
class Config:
    """Configuration options for the Lightning MNIST example.

    Args:
        data_dir : The path to the directory where the MNIST dataset is stored. Defaults to the value of
            the 'PATH_DATASETS' environment variable or '.' if not set.
        save_dir : The path to the directory where the training logs will be saved. Defaults to 'logs/'.
        batch_size : The batch size to use during training. Defaults to 256 if a GPU is available,
            or 64 otherwise.
        max_epochs : The maximum number of epochs to train the model for. Defaults to 3.
        accelerator : The accelerator to use for training. Can be one of "cpu", "gpu", "tpu", "ipu", "auto".
        devices : The number of devices to use for training. Defaults to 1.
    """

    data_dir: str = Path("data/MNIST")
    save_dir: str = "logs/"
    batch_size: int = 256 if torch.cuda.is_available() else 64
    max_epochs: int = 3
    accelerator: str = "auto"
    devices: int = 1

Define a Lightning module.

In [ ]:
class MNISTModel(pl.LightningModule):
    """A PyTorch Lightning module for classifying images in the MNIST dataset.

    Attributes:
        l1 : A linear layer that maps input features to output features.

    Methods:
        forward(x):
            Performs a forward pass through the model.

        training_step(batch, batch_nb):
            Defines a single training step for the model.

        configure_optimizers():
            Configures the optimizer to use during training.
    """

    def __init__(self):
        """Initializes a new instance of the MNISTModel class."""
        super().__init__()
        self.l1 = torch.nn.Linear(28 * 28, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Performs a forward pass through the model.

        Args:
            x : The input tensor to pass through the model.

        Returns:
            activated : The output tensor produced by the model.

        """
        flattened = x.view(x.size(0), -1)
        hidden = self.l1(flattened)
        activated = torch.relu(hidden)

        return activated

    def training_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_nb: int,
    ) -> torch.Tensor:
        """Defines a single training step for the model.

        Args:
            batch: A tuple containing the input and target tensors for the batch.
            batch_nb: The batch number.

        Returns:
            torch.Tensor: The loss value for the current batch.

        """
        x, y = batch
        loss = F.cross_entropy(self(x), y)
        return loss

    def configure_optimizers(self) -> torch.optim.Optimizer:
        """Configures the optimizer to use during training.

        Returns:
            torch.optim.Optimizer: The optimizer to use during training.

        """
        return torch.optim.Adam(self.parameters(), lr=0.02)

Train the model.

In [ ]:
config = Config()

# Init our model
mnist_model = MNISTModel()

# Init DataLoader from MNIST Dataset
train_ds = MNIST(
    config.data_dir,
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

# Create a dataloader
train_loader = DataLoader(train_ds, batch_size=config.batch_size)

# Initialize a trainer
trainer = pl.Trainer(
    accelerator=config.accelerator,
    devices=config.devices,
    max_epochs=config.max_epochs,
)

# Train the model ⚡
trainer.fit(mnist_model, train_loader)